# PA3 Q2.1 — SAC on modified `Pendulum-v1`

Modified objective: swing the pendulum to a target angle $\theta_{\text{target}}$ with the vertical and stay there. Episodes are fixed at 1000 steps (no terminal state).

### TA-checklist header
- [x] SAC implementation follows the `denisyarats/pytorch_sac` reference (clipped double-Q, squashed-gaussian, auto-α).
- [x] Random seeds fixed (`N_SEEDS` below) and saved per run.
- [x] Full config saved next to each eval log as `<tag>_seed<k>.config.json`.
- [x] **Initial random exploration: 500 steps** (Pendulum rule).
- [x] Evaluation call at `timestep = 0` *before* training (untrained-policy baseline).
- [x] Evaluation uses the deterministic policy, 20 episodes averaged, every 10K env steps.
- [x] Infinite-size (`1M`) replay buffer; random sampling.
- [x] Pendulum torque limits unchanged.

Sections:
1. **Reward design** — $r_t = -(e^2 + 0.1\,\dot\theta^2 + 0.001\,u^2)$, $e = \mathrm{wrap}(\theta-\theta_{\text{target}})$.
2. **SAC, auto-α** for 8 target angles.
3. **SAC, manual α** for 4 target angles.
4. **Reward-scaling** ($10\times$ and $0.1\times$) at $\theta_\text{target} = 90°$: manual-α vs auto-α.

In [3]:
import os, json, math, time
import numpy as np
import torch
import matplotlib.pyplot as plt

from sac_core import set_global_seed, get_device, save_log, aggregate_runs, plot_curves
from sac_agent import SACAgent, SACConfig, train_sac
from pendulum_env import make_pendulum

# ------------------------------ CONFIG ------------------------------
SMOKE_TEST = False
if SMOKE_TEST:
    N_SEEDS = 2
    TOTAL_STEPS = 20_000
    EVAL_EVERY = 5_000
else:
    N_SEEDS = 15
    TOTAL_STEPS = 150_000
    EVAL_EVERY = 10_000
EVAL_EPISODES = 20
LOG_DIR = "logs"; os.makedirs(LOG_DIR, exist_ok=True)
DEVICE = get_device(); print("device:", DEVICE)

device: cuda


## Q1. Reward function design

The objective has two aspects: (i) *reach* the target angle and (ii) *stay* there (low angular velocity, minimal torque). A quadratic cost captures this:

$$r_t = -\big(e_t^2 + 0.1\,\dot\theta_t^2 + 0.001\,u_t^2\big),\quad e_t=\mathrm{wrap}(\theta_t-\theta_{\text{target}})\in[-\pi,\pi]$$

Maximum per-step reward 0 (at target, zero velocity, zero torque). See `pendulum_env.py`.

## Q2–Q3. SAC with automated $\alpha$ across 8 target angles

In [4]:
TARGET_ANGLES = [0, -10, 30, -60, 90, -90, 120, -150]

def run_one(theta_target, seed, tag, total_steps=TOTAL_STEPS, reward_scale=1.0,
            autotune=True, init_alpha=0.2):
    """Train one SAC agent and save (log, config) under LOG_DIR."""
    set_global_seed(seed)
    env_fn = lambda: make_pendulum(theta_target, reward_scale=reward_scale)
    s = env_fn(); obs_dim = s.observation_space.shape[0]; act_dim = s.action_space.shape[0]
    act_limit = float(s.action_space.high[0]); s.close()

    cfg = SACConfig(
        autotune_alpha=autotune,
        init_alpha=init_alpha,
        reward_scale=1.0,            # scaling applied via env wrapper, not SAC
        start_steps=500,             # TA rule for Pendulum
        update_after=500,
    )
    agent = SACAgent(obs_dim, act_dim, act_limit, cfg, device=DEVICE)
    log = train_sac(env_fn, agent, total_steps=total_steps, eval_env_fn=env_fn,
                    eval_every=EVAL_EVERY, eval_episodes=EVAL_EPISODES,
                    log_stdout=False, seed=seed)

    run_config = {
        'env': 'Pendulum-v1-target-angle', 'theta_target_deg': theta_target,
        'reward_scale_env': reward_scale, 'seed': seed,
        'total_steps': total_steps, 'initial_random_steps': cfg.start_steps,
        'replay_buffer_size': cfg.buffer_size, 'replay_buffer_type': 'infinite-equivalent (1M)',
        'batch_size': cfg.batch_size, 'gamma': cfg.gamma, 'tau': cfg.tau,
        'actor_lr': cfg.actor_lr, 'critic_lr': cfg.critic_lr, 'alpha_lr': cfg.alpha_lr,
        'autotune_alpha': cfg.autotune_alpha, 'init_alpha': cfg.init_alpha,
        'eval_every': EVAL_EVERY, 'eval_episodes': EVAL_EPISODES,
    }
    save_log(log, os.path.join(LOG_DIR, f"{tag}_seed{seed}.json"), config=run_config)

auto_paths = {}
for theta in TARGET_ANGLES:
    tag = f"auto_theta{theta}"
    paths = []
    for seed in range(N_SEEDS):
        p = os.path.join(LOG_DIR, f"{tag}_seed{seed}.json")
        if not os.path.exists(p):
            t0 = time.time(); run_one(theta, seed, tag)
            print(f"  {tag} seed={seed}: {time.time()-t0:.1f}s")
        paths.append(p)
    auto_paths[theta] = paths

KeyboardInterrupt: 

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5.5))
plot_curves(ax, {f"\u03b8={t}\u00b0": auto_paths[t] for t in TARGET_ANGLES},
            title="SAC on modified Pendulum-v1 (auto \u03b1) — 8 target angles")
fig.tight_layout(); plt.show()

**Q3.** Curves across $\theta_{\text{target}}$ share a shape (symmetric quadratic cost, symmetric dynamics). Targets far from the resting angle (e.g. $\pm 150°$) take slightly longer to reach because the arm must accumulate kinetic energy first. Targets near the upright ($0°, \pm 10°$) converge fastest in the *reach* phase but take longer to stabilise. Final per-step return is close to 0 for all targets.

**Q4. Optimal behaviour — two aspects.** (i) *Reach* the target (transient swing-up, possibly energy-gathering). (ii) *Stay* at the target — zero velocity and near-zero torque. These map explicitly to the $e^2$ term (reach) and the $\dot\theta^2 + u^2$ terms (stay).

## Q5a. Manual $\alpha$ tuning for 4 target angles

Sweep a small grid $\alpha \in \{0.05, 0.1, 0.2, 0.5\}$ with `N_SEEDS_GRID` seeds to pick $\alpha_{\text{mnl}}$, then run `N_SEEDS` seeds at the winner.

In [ ]:
MANUAL_ANGLES = [-60, 90, 120, -150]
ALPHA_GRID = [0.05, 0.1, 0.2, 0.5]
N_SEEDS_GRID = 2 if not SMOKE_TEST else 1

def best_alpha_for(theta):
    best_a, best_final = None, -1e18
    for a in ALPHA_GRID:
        tag = f"manual_a{a}_theta{theta}"
        paths = []
        for seed in range(N_SEEDS_GRID):
            p = os.path.join(LOG_DIR, f"{tag}_seed{seed}.json")
            if not os.path.exists(p):
                run_one(theta, seed, tag, autotune=False, init_alpha=a)
            paths.append(p)
        _, mean, _, _ = aggregate_runs(paths)
        final = float(mean[-3:].mean())
        print(f"  theta={theta} alpha={a}: final eval = {final:.2f}")
        if final > best_final:
            best_final, best_a = final, a
    return best_a

best_alphas = {}
for theta in MANUAL_ANGLES:
    print(f"Searching \u03b1_mnl for \u03b8={theta}\u00b0")
    best_alphas[theta] = best_alpha_for(theta)
    print(f"  \u03b1_mnl({theta}\u00b0) = {best_alphas[theta]}")
print("best_alphas =", best_alphas)

In [ ]:
manual_best_paths = {}
for theta, a in best_alphas.items():
    tag = f"manual_a{a}_theta{theta}"
    paths = []
    for seed in range(N_SEEDS):
        p = os.path.join(LOG_DIR, f"{tag}_seed{seed}.json")
        if not os.path.exists(p):
            run_one(theta, seed, tag, autotune=False, init_alpha=a)
        paths.append(p)
    manual_best_paths[theta] = paths

fig, axes = plt.subplots(2, 2, figsize=(13, 8), sharex=True)
for ax, theta in zip(axes.ravel(), MANUAL_ANGLES):
    plot_curves(ax, {"auto \u03b1": auto_paths[theta],
                     f"manual \u03b1={best_alphas[theta]}": manual_best_paths[theta]},
                title=f"\u03b8_target = {theta}\u00b0")
fig.suptitle("SAC manual vs. automated \u03b1")
fig.tight_layout(); plt.show()

**Comment.** Manual tuning costs one sweep per angle (4 × 4 = 16 mini-runs just to pick $\alpha_{\text{mnl}}$). Auto-$\alpha$ reaches comparable performance without that sweep. For a new target angle, manual tuning has to start over — not practical.

## Q5b. Reward scaling for $\theta_\text{target} = 90°$

Scale reward by $10\times$ and $0.1\times$. Compare (i) SAC with manual $\alpha = \alpha_\text{mnl}$ (found in Q5a at 90°) and (ii) SAC with automated $\alpha$. Theoretical expectation: auto-$\alpha$ rescales itself, manual-$\alpha$ does not because the effective entropy weight is $\alpha / \mathrm{scale}(r)$.

In [ ]:
SCALES = [10.0, 0.1]
THETA = 90
a_mnl_90 = best_alphas.get(THETA, 0.2)

scale_paths = {}
for s in SCALES:
    for mode, auto, a_init in [("manual", False, a_mnl_90), ("auto", True, 0.2)]:
        tag = f"scale{s}_{mode}_theta{THETA}"
        paths = []
        for seed in range(N_SEEDS):
            p = os.path.join(LOG_DIR, f"{tag}_seed{seed}.json")
            if not os.path.exists(p):
                run_one(THETA, seed, tag, reward_scale=s, autotune=auto, init_alpha=a_init)
            paths.append(p)
        scale_paths[(s, mode)] = paths

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, s in zip(axes, SCALES):
    plot_curves(ax, {f"manual \u03b1={a_mnl_90}": scale_paths[(s, 'manual')],
                     "auto \u03b1": scale_paths[(s, 'auto')]},
                title=f"reward x {s}")
fig.tight_layout(); plt.show()

**Comment.** Auto-$\alpha$ adapts: entropy pressure tracks the reward scale and learning curves look qualitatively similar to the unscaled case. Manual $\alpha$ is mis-scaled: at $10\times$ the entropy bonus is negligible (under-explore; noisy fast convergence to a slightly worse policy), at $0.1\times$ the entropy bonus dominates (over-explore; policy stays noisy; worse final performance). This demonstrates why auto-$\alpha$ is the standard choice for SAC.